In [ ]:
import os
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv

# PRE-REQUISITES:
# generate the client ID and secret for the service principal separately in Databricks and set them in the math_agent.env file
# grant the SP CAN_QUERY permission on the endpoint
load_dotenv(dotenv_path="math_agent.env", override=True)

w = WorkspaceClient(
    host="https://dbc-564fb500-5a75.cloud.databricks.com/",
    client_id=os.environ.get("DATABRICKS_CLIENT_ID"),
    client_secret=os.environ.get("DATABRICKS_CLIENT_SECRET")
)

In [30]:
import os
import requests
import pandas as pd
import json


def create_tf_serving_json(data: dict):
    """
    Add the mlflow required "input" wrapper
    """
    return {
        "input": [data]
    }



def get_oauth_token(databricks_host, client_id, client_secret):
    """
    Get OAuth token using service principal credentials
    
    Args:
        databricks_host: Your Databricks workspace URL (e.g., "https://dbc-564fb500-5a75.cloud.databricks.com")
        client_id: Service principal client ID
        client_secret: Service principal client secret
    
    Returns:
        dict: Token response containing access_token, token_type, expires_in
    """
    # Construct the token endpoint URL
    token_url = f"{databricks_host}/oidc/v1/token"
    
    # Prepare the request
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }
    
    # Method 1: Using Basic Authentication (recommended)
    auth = (client_id, client_secret)
    data = {
        "grant_type": "client_credentials",
        "scope": "all-apis"  # or specify specific scopes
    }
    
    response = requests.post(
        url=token_url,
        headers=headers,
        auth=auth,
        data=data
    )
    
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Token request failed: {response.status_code} - {response.text}")

def score_model(query:str, databricks_host, endpoint_name, client_id, client_secret):
    """
    Score a model deployed to a Databricks Serving Endpoint.

    Args:
        query: Input query string (e.g., "what is the volume of a cylinder with height of 5 and radius of 2?")
        databricks_host: Your Databricks workspace URL (e.g., "https://dbc-564fb500-5a75.cloud.databricks.com")
        endpoint_name: Name of the serving endpoint
        client_id: Service principal client ID
        client_secret: Service principal client secret
    
    Returns:
        dict: Model prediction response

    """
    
    url = f"{databricks_host}/serving-endpoints/{endpoint_name}/invocations"    
    token_response = get_oauth_token(databricks_host, client_id, client_secret)
    access_token = token_response["access_token"]
    
    headers = {
        "Authorization": f'Bearer {access_token}',
        "Content-Type": "application/json",
    }

    dataset = {"role": "user", "content": query}
    ds_dict = (
        {"dataframe_split": dataset.to_dict(orient="split")}
        if isinstance(dataset, pd.DataFrame)
        else create_tf_serving_json(dataset)
    )
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method="POST", headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(
            f"Request failed with status {response.status_code}, {response.text}"
        )
    return response.json()

In [28]:
data = {
    "role": "user",
    "content": "what is the volume of a cylinder with height of 5 and radius of 2?",
}

ds_dict = (
    {"dataframe_split": data.to_dict(orient="split")}
    if isinstance(data, pd.DataFrame)
    else create_tf_serving_json(data)
)

data_json = json.dumps(ds_dict, allow_nan=True)

data_json

'{"input": [{"role": "user", "content": "what is the volume of a cylinder with height of 5 and radius of 2?"}]}'

In [20]:
# Usage example
try:
    databricks_host = "https://dbc-564fb500-5a75.cloud.databricks.com"
    client_id = os.environ.get("DATABRICKS_CLIENT_ID")
    client_secret = os.environ.get("DATABRICKS_CLIENT_SECRET")
    
    token_response = get_oauth_token(databricks_host, client_id, client_secret)
    
    access_token = token_response["access_token"]
    print(f"Access token obtained: {access_token[:20]}...")
    print(f"Token type: {token_response['token_type']}")
    print(f"Expires in: {token_response['expires_in']} seconds")
    
except Exception as e:
    print(f"Error obtaining token: {e}")

Access token obtained: eyJraWQiOiI2NDZiZWZk...
Token type: Bearer
Expires in: 3600 seconds


In [31]:
query = "what is the volume of a cylinder with height of 15 and radius of 5?"

response = score_model(
    query,
    databricks_host="https://dbc-564fb500-5a75.cloud.databricks.com",
    endpoint_name="math_agent-endpoint",
    client_id=os.environ.get("DATABRICKS_CLIENT_ID"),
    client_secret=os.environ.get("DATABRICKS_CLIENT_SECRET")
)
response

{'object': 'response',
 'output': [{'type': 'function_call',
   'id': 'lc_run--019c48ba-ae15-7e02-a561-503647e32e48-0',
   'call_id': 'call_VLCDqAL65dhi2drLb23iQ4cp',
   'name': 'cylinder_volume_tool',
   'arguments': '{"height": 15, "radius": 5}'},
  {'type': 'function_call_output',
   'call_id': 'call_VLCDqAL65dhi2drLb23iQ4cp',
   'output': '1178.0972450961724'},
  {'type': 'message',
   'id': 'lc_run--019c48ba-b17a-7190-a4f2-a4742948009a-0',
   'content': [{'text': 'The volume of the cylinder is approximately 1178.10 cubic units.',
     'type': 'output_text'}],
   'role': 'assistant'}],
 'id': 'a9308490-9a87-4ac9-888b-9b50bdbf11b4'}